In [1]:
import sys
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from hyperopt import fmin, tpe, hp, Trials
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings("ignore")

In [2]:
# --------------- Set up project root path  --------------- #
project_folder_name = "MFC2024" # Set this to the name of your project root folderS
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == project_folder_name), None)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# --------------- Import custom modules  --------------- #
from model.dualscale import PEMFC
from model.coefficients import *
from modules.signals import generate_step_load
from config.initialize import *
from config.settings import *

In [3]:
# --- Global parameters ---
tstart     = 0.0      # start of each period                    [s]
tend       = 6.0    # end of each period                      [s]
i_low      = 0.002e4      # baseline current density                [A/m^2]
i_high     = 1.2e4    # plateau current density                 [A/m^2]
tau_switch = 1.0     # time (within a period) the ramp BEGINS  [s]
t_switch   = 3.0      # effective ramp duration                 [s]

step_load = generate_step_load(tstart, tend, i_low, i_high, tau_switch, t_switch)

In [4]:
operating_inputs["current_density"] = step_load
operating_inputs["Phi_c_des"] = 0.2
model = PEMFC(param=parameters, operating_inputs=operating_inputs,
                           variable_names=solver_variable_names, flux_names=solver_flux_names)
solution_init = init_x(operating_inputs, parameters)
sol = solve_ivp(fun = model.dxdt, y0=solution_init, t_span=(0, 200), method='BDF', max_step=0.1)

In [5]:
sol = solve_ivp(fun = model.dxdt, y0=solution_init, t_span=(0, 5000 * 6 *7), method='BDF', max_step=0.1)

ValueError: array must not contain infs or NaNs